# Wait for data-platform ingestion

A tool run can report `Completed` before the data platform has finished ingesting its
results. During that window the data-platform execution record reads `DataIngesting`.

`client.executions.wait_for_ingestion` blocks until ingestion has finished and returns the
data-platform execution record. It is woken by server-sent events rather than polling, so
it returns within seconds of the record changing.

In [ ]:
from deeporigin import projects
from deeporigin.platform import DeepOriginClient

client = DeepOriginClient.from_disk()
projects.create("python-client-dev", client=client)

## Start an import

Upload a small SMILES CSV and import it with `deeporigin.import-dataset`.

In [ ]:
import tempfile
import time
from pathlib import Path

smiles = {
    "ethanol": "CCO",
    "propanol": "CCCO",
    "benzene": "c1ccccc1",
    "phenol": "Oc1ccccc1",
    "toluene": "Cc1ccccc1",
}
local_csv = Path(tempfile.mkdtemp()) / "ligands.csv"
local_csv.write_text("name,smiles\n" + "".join(f"{n},{s}\n" for n, s in smiles.items()))

remote_csv = f"notebooks/wait-for-ingestion/{time.strftime('%Y%m%dT%H%M%S')}/ligands.csv"
client.files.upload(local_csv, remote_csv)

execution = client.executions.create(
    tool_key="deeporigin.import-dataset",
    tool_version="4.0.0",
    data={
        "inputs": {
            "csv_path": remote_csv,
            "mapper": [
                {"field": "smiles", "json-path": "ligands.smiles"},
                {"field": "name", "json-path": "ligands.name"},
            ],
            "database_key": "deeporigin.attributes_catalog",
            "database_version": "1.0.0",
        },
        "outputs": {},
        "metadata": {},
    },
)
execution_id = execution["executionId"]
execution_id, execution["status"]

## Wait for ingestion

`project_id` defaults to `client.project_id`, and `timeout` to one hour. A run that never
finishes ingesting raises `TimeoutError`.

In [ ]:
start = time.monotonic()
row = client.executions.wait_for_ingestion(execution_id)
print(f"returned after {time.monotonic() - start:.1f}s")
{key: row[key] for key in ("display_id", "status", "completed_at")}

The record is returned for any finished status, including `Failed` and `Cancelled`,
so check `status` before using the results:

In [ ]:
if row["status"] != "Completed":
    raise RuntimeError(f"Import ended with status {row['status']}")

## An already-finished run

Waiting on a run whose ingestion has already finished returns straight away.

In [ ]:
start = time.monotonic()
row = client.executions.wait_for_ingestion(execution_id)
print(f"returned after {time.monotonic() - start:.1f}s")
row["status"]